# 🎯 Entrenamiento de Modelo de Detección de Objetos (PPE) con YOLOv8

Este cuaderno es una guía completa y detallada para entrenar un modelo de Inteligencia Artificial capaz de reconocer **Equipos de Protección Personal (PPE)**. Seguiremos una metodología de Ciencia de Datos para garantizar un modelo robusto y preciso.

---

## 1. 📖 Contexto e Historia de YOLO
**YOLO (You Only Look Once)** revolucionó la visión por computadora al proponer que la detección de objetos se realizara en una sola pasada de la red neuronal, a diferencia de métodos anteriores que usaban ventanas deslizantes o propuestas de región.

### Evolución:
*   **YOLOv1 - v3:** Las bases del tiempo real.
*   **YOLOv5:** Popularizó la facilidad de uso y entrenamiento.
*   **YOLOv8 (Ultralytics):** La versión que utilizaremos hoy. Es un modelo SOTA (State of the Art) que permite detección, segmentación y clasificación en un solo framework optimizado.

---

## 2. ⚖️ Licencias y Modelos
Utilizaremos la arquitectura **YOLOv8 Nano (`yolov8n.pt`)**. 
*   **¿Por qué Nano?** Es extremadamente ligero, lo que permite que nuestra aplicación de Streamlit funcione rápidamente incluso en hardware modesto (CPU), manteniendo una excelente precisión tras el entrenamiento.
*   **Licencia:** Ultralytics usa AGPL-3.0 para código abierto y licencias comerciales para empresas.

---

## 3. ⚙️ Instalación y Configuración del Entorno
Antes de comenzar, debemos preparar nuestro entorno de trabajo. Instalaremos `ultralytics` (que contiene YOLOv8) y `roboflow` (por si necesitamos descargar datos).


In [2]:
import torch
import os
import shutil
from IPython.display import display, Image, clear_output

# --- NUEVO: SOPORTE PARA GOOGLE DRIVE ---
# Intentar montar Google Drive para persistencia de datos
DRIVE_MOUNT_PATH = "/content/drive"
DRIVE_PROJECT_PATH = f"{DRIVE_MOUNT_PATH}/MyDrive/UNAB_DeteccionObjetos"

try:
    from google.colab import drive
    print("📂 Detectado entorno Google Colab. Intentando montar Google Drive...")
    drive.mount(DRIVE_MOUNT_PATH)
    os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
    print(f"✅ Google Drive vinculado. Los respaldos se guardarán en: {DRIVE_PROJECT_PATH}")
    USAR_DRIVE = True
except ImportError:
    print("ℹ️ No se detectó entorno Colab o no se pudo montar Drive. Se usará almacenamiento local.")
    USAR_DRIVE = False
except Exception as e:
    print(f"⚠️ Error al montar Drive: {e}. Se continuará con almacenamiento local.")
    USAR_DRIVE = False

# Verificar si tenemos una GPU disponible (Crucial para entrenar rápido)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Trabajando en: {device}")
if torch.cuda.is_available():
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")

# Instalación de librerías (Necesario en Google Colab)
!pip install ultralytics roboflow opencv-python --quiet

from ultralytics import YOLO
print("✅ Librerías cargadas correctamente")


📂 Detectado entorno Google Colab. Intentando montar Google Drive...
⚠️ Error al montar Drive: mount failed. Se continuará con almacenamiento local.
🚀 Trabajando en: cuda
GPU detectada: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 99.2 MB/s eta 0:00:00ta 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-setti

---

## 4. 📁 Carga y Preparación del Dataset (Automática)
Como estás ejecutando este cuaderno desde un **IDE** conectado a **Colab**, hemos automatizado la carga para que no dependas del selector de archivos del navegador, el cual suele fallar en entornos de desarrollo locales.

### Paso 4.1: Detección y Adquisición del Dataset
El código verificará si `ppe.zip` ya existe. Si no está, o si el archivo está dañado, lo descargaremos automáticamente desde el repositorio de Roboflow para garantizar que el entrenamiento pueda iniciar sin interrupciones.


In [3]:
import os
import zipfile
import shutil
from roboflow import Roboflow

# Configuración
archivo_zip = "ppe.zip"
folder_dataset = "dataset_ppe"

def adquirir_dataset():
    # 0. Verificar si el dataset ya está extraído y listo (Ideal para procesamiento local)
    if os.path.exists(folder_dataset) and os.path.isdir(folder_dataset):
        # Comprobamos si tiene la estructura mínima de YOLO para evitar descargas innecesarias
        if os.path.exists(os.path.join(folder_dataset, "train")):
            print(f"✅ El dataset ya existe en '{folder_dataset}'. Omitiendo descarga.")
            return False

    # 1. Verificar si el archivo ZIP ya existe localmente
    if os.path.exists(archivo_zip):
        print(f"🔍 Se encontró '{archivo_zip}' en el directorio actual.")
        
        # Verificar si es un HTML corrupto (error común al descargar mal de Drive)
        with open(archivo_zip, 'rb') as f:
            cabeza = f.read(100).lower()
            if b'<!doctype html>' in cabeza or b'<html' in cabeza:
                print("⚠️ El archivo local es un HTML corrupto. Procediendo a descarga automática...")
            else:
                print("✅ El archivo local parece válido. Usaremos este.")
                return True

    # 2. Si no existe o es inválido, descargar de Roboflow (Plan B Automático)
    print("📡 Iniciando descarga automática desde Roboflow...")
    try:
        # Usamos la API Key del cuaderno de referencia para asegurar compatibilidad
        rf = Roboflow(api_key="dSMfDD4uPaMCKEoGOP5q")
        project = rf.workspace("cicatriz").project("ppe-factory-bmdcj-alnpk")
        version = project.version(1)
        # Descargamos en formato YOLOv8
        dataset = version.download("yolov8")
        
        # Mover a la ubicación esperada
        src_path = os.path.abspath(dataset.location)
        dest_path = os.path.abspath(folder_dataset)
        
        if src_path != dest_path:
            if os.path.exists(dest_path):
                shutil.rmtree(dest_path)
            shutil.move(src_path, dest_path)
            print(f"✅ Dataset descargado y movido a '{folder_dataset}'")
        else:
            print(f"✅ Dataset descargado directamente en '{folder_dataset}'")
            
        return False # Ya está extraído por Roboflow
    except Exception as e:
        print(f"❌ Error en la descarga automática: {e}")
        return False

necesita_extraccion = adquirir_dataset()


📡 Iniciando descarga automática desde Roboflow...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ppe-factory-1 in yolov8:: 100%|██████████| 21608/21608 [00:05<00:00, 3838.37it/s] 

✅ Dataset descargado y movido a 'dataset_ppe'


### Paso 4.2: Extracción y Validación (Si aplica)
Si el archivo ya estaba presente como un ZIP válido, procederemos a su extracción manual.


In [4]:
if necesita_extraccion:
    if os.path.exists(archivo_zip):
        # Limpiar carpeta si ya existe
        if os.path.exists(folder_dataset):
            shutil.rmtree(folder_dataset)
        
        try:
            with zipfile.ZipFile(archivo_zip, 'r') as zip_ref:
                zip_ref.extractall(folder_dataset)
            print(f"✅ Extracción completada en '{folder_dataset}'")
        except Exception as e:
            print(f"❌ Error al extraer: {e}")

# Verificación final de la estructura (Ciencia de Datos)
if os.path.exists(folder_dataset):
    print("\n📊 Resumen del Dataset:")
    for split in ['train', 'valid', 'test']:
        img_path = os.path.join(folder_dataset, split, 'images')
        if os.path.exists(img_path):
            n = len(os.listdir(img_path))
            print(f"  - {split.capitalize()}: {n} imágenes.")
else:
    print("⚠️ No se pudo preparar el dataset. Revisa los mensajes anteriores.")



📊 Resumen del Dataset:
  - Train: 9770 imágenes.
  - Valid: 742 imágenes.
  - Test: 286 imágenes.


### Paso 4.3: Verificación de la estructura `data.yaml`
YOLOv8 necesita un archivo llamado `data.yaml` que le dice dónde están las imágenes y cuáles son los nombres de las clases (Vest, Safety Shoe, Mask, Helmet, Goggles, Gloves).


In [5]:
import yaml

# Intentamos localizar el archivo data.yaml dentro del dataset extraído
def buscar_y_corregir_yaml(directorio):
    path_encontrado = None
    for root, dirs, files in os.walk(directorio):
        if 'data.yaml' in files:
            path_encontrado = os.path.join(root, 'data.yaml')
            break
    
    if path_encontrado:
        # Corregir las rutas dentro del YAML para que sean absolutas
        # Esto evita errores cuando YOLO busca las carpetas desde diferentes directorios
        try:
            with open(path_encontrado, 'r') as f:
                data = yaml.safe_load(f)
            
            base_dir = os.path.dirname(os.path.abspath(path_encontrado))
            
            # Ajustamos las rutas a formato absoluto para robustez local
            data['train'] = os.path.join(base_dir, 'train', 'images')
            data['val'] = os.path.join(base_dir, 'valid', 'images')
            if 'test' in data:
                data['test'] = os.path.join(base_dir, 'test', 'images')
                
            with open(path_encontrado, 'w') as f:
                yaml.dump(data, f, default_flow_style=False)
        except Exception as e:
            print(f"⚠️ Error al corregir YAML: {e}. Se usará el archivo original.")
            
        return path_encontrado
    return None

path_yaml = buscar_y_corregir_yaml(folder_dataset)
if path_yaml:
    print(f"✅ Archivo de configuración encontrado en: {path_yaml}")
    # Mostrar contenido actualizado
    with open(path_yaml, 'r') as f:
        print("\nContenido de data.yaml:")
        print(f.read())
else:
    print("⚠️ No se encontró data.yaml. Es posible que el ZIP no tenga el formato estándar de YOLOv8.")


✅ Archivo de configuración encontrado en: dataset_ppe/data.yaml

Contenido de data.yaml:
names:
- boots
- earmuffs
- glasses
- gloves
- helmet
- person
- vest
nc: 7
roboflow:
  license: CC BY 4.0
  project: ppe-factory-bmdcj-vb0jb
  url: https://universe.roboflow.com/manuel-ybi42/ppe-factory-bmdcj-vb0jb/dataset/1
  version: 1
  workspace: manuel-ybi42
test: /content/dataset_ppe/test/images
train: /content/dataset_ppe/train/images
val: /content/dataset_ppe/valid/images



---

## 5. 🚀 Entrenamiento del Modelo (Hacia el >85% de Precisión)
Para lograr una precisión superior al 85% (mAP@50), configuraremos varios hiperparámetros:
*   **Epochs (Épocas):** Entrenaremos por 50 a 100 épocas. El modelo se detendrá antes si deja de mejorar (Early Stopping).
*   **Imgsz (Tamaño):** Usaremos 640 píxeles, que es el estándar para YOLOv8.
*   **Patience (Paciencia):** Esperaremos 20 épocas sin mejora antes de parar.
*   **Resiliencia Total:** 
    1. El progreso se guarda automáticamente cada **2 épocas**.
    2. Si Colab se desconecta, puedes subir tu archivo `last.pt` local para continuar desde donde quedaste.
    3. Al finalizar (o si detienes manualmente), se descargará el último checkpoint a tu PC.


In [6]:
# Lógica para REANUDAR entrenamiento desde local en Colab
checkpoint_dir = 'runs/detect/ppe_training_v1/weights'
checkpoint_path = os.path.join(checkpoint_dir, 'last.pt')

try:
    from google.colab import files
    import os
    
    if not os.path.exists(checkpoint_path):
        print("❓ ¿Tienes un respaldo de 'last.pt' en tu PC y quieres reanudar el entrenamiento?")
        subir = input("¿Deseas subir el archivo last.pt? (s/n): ").lower()
        if subir == 's':
            uploaded = files.upload()
            if 'last.pt' in uploaded:
                os.makedirs(checkpoint_dir, exist_ok=True)
                with open(checkpoint_path, 'wb') as f:
                    f.write(uploaded['last.pt'])
                print(f"✅ Archivo 'last.pt' subido y colocado en {checkpoint_path}")
except ImportError:
    pass

# 1. Definir la función de descarga automática y respaldo en Drive
def backup_callback(trainer):
    """Callback que se ejecuta cada vez que YOLO guarda los pesos (.pt)"""
    try:
        import os
        import shutil
        # Ruta al último checkpoint guardado en el entorno de Colab
        checkpoint_path = os.path.join(trainer.save_dir, 'weights', 'last.pt')
        
        if os.path.exists(checkpoint_path):
            # Opción A: Respaldo en Google Drive (Más confiable)
            if USAR_DRIVE:
                drive_dest = os.path.join(DRIVE_PROJECT_PATH, 'last.pt')
                shutil.copy(checkpoint_path, drive_dest)
                print(f"\n☁️ Respaldo en Drive actualizado en: {drive_dest}")

            # Opción B: Descarga al PC (Sujeto a permisos del navegador)
            from google.colab import files
            print(f"💾 Iniciando descarga local de 'last.pt'...")
            files.download(checkpoint_path)
    except ImportError:
        pass # No estamos en Colab
    except Exception as e:
        print(f"⚠️ Error en backup_callback: {e}")

# Cargar el modelo para entrenar o reanudar
# Intentar buscar el checkpoint primero en Drive, luego local
if USAR_DRIVE and os.path.exists(os.path.join(DRIVE_PROJECT_PATH, 'last.pt')):
    print("🔄 Encontrado checkpoint en Google Drive. Sincronizando...")
    os.makedirs(checkpoint_dir, exist_ok=True)
    shutil.copy(os.path.join(DRIVE_PROJECT_PATH, 'last.pt'), checkpoint_path)

if os.path.exists(checkpoint_path):
    print(f"🔄 Detectado checkpoint listo. Cargando {checkpoint_path}...")
    model = YOLO(checkpoint_path)
    resume_train = True
else:
    print("🆕 No se detectó checkpoint previo. Iniciando desde el modelo base YOLOv8n.")
    model = YOLO('yolov8n.pt')
    resume_train = False

# 2. Registrar el callback en el modelo antes de entrenar
model.add_callback("on_model_save", backup_callback)

if path_yaml:
    # Entrenamiento con guardado de checkpoints cada 2 épocas
    results = model.train(
        data=path_yaml,
        epochs=60,       # Ajustar según necesidad
        imgsz=640,       
        patience=4,
        batch=16,        
        name='ppe_training_v1',
        plots=True,      
        save=True,       
        save_period=2,   # Esto disparará el callback cada 2 épocas
        resume=resume_train 
    )
    print("✅ Entrenamiento finalizado.")
else:
    print("❌ No se puede iniciar el entrenamiento sin el archivo data.yaml.")


❓ ¿Tienes un respaldo de 'last.pt' en tu PC y quieres reanudar el entrenamiento?
🆕 No se detectó checkpoint previo. Iniciando desde el modelo base YOLOv8n.
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_ppe/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, mome

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/60      2.87G      1.282      1.342      1.357         85        640: 100% ━━━━━━━━━━━━ 611/611 3.6it/s 2:51<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 3.4it/s 7.1s0.3s
                   all        742       3828      0.617      0.593      0.611      0.354
💾 Iniciando descarga local de 'last.pt'...


<IPython.core.display.Javascript object>

...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/60      2.87G      1.268       1.22      1.349         74        640: 100% ━━━━━━━━━━━━ 611/611 3.7it/s 2:45<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 3.2it/s 7.6s0.2s
                   all        742       3828      0.654      0.606       0.63      0.368
💾 Iniciando descarga local de 'last.pt'...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/60      2.87G      1.261      1.152      1.346        121        640: 50% ━━━━━━────── 308/611 4.5it/s 1:24<1:089


KeyboardInterrupt: 

---

## 6. 📊 Evaluación Detallada y Validación de Métricas
Como científicos de datos, no podemos confiar solo en que el entrenamiento "termine". Debemos validar la calidad del modelo usando métricas estándar de la industria.

### ¿Qué es el mAP? (Mean Average Precision)
En detección de objetos, no usamos el "Accuracy" tradicional porque este no considera qué tan bien centrada está la caja (Bounding Box). En su lugar usamos el **mAP**:
*   **mAP50:** Es nuestra métrica principal. Mide la precisión cuando el modelo acierta al menos el 50% del área del objeto.
*   **Meta:** Para este proyecto, buscamos un **mAP50 > 0.85 (85%)**.


In [ ]:
# Validar el modelo con el set de validación
if path_yaml:
    # Usamos el set de validación separado (valid)
    metrics = model.val() 
    
    print("\n📊 RESULTADOS DE LA VALIDACIÓN:")
    print(f"mAP@50 (Precisión Media): {metrics.box.map50:.4f}")
    print(f"mAP@50-95 (Rigor Extremo): {metrics.box.map:.4f}")
    
    if metrics.box.map50 > 0.85:
        print("🏆 ¡Excelente! El modelo superó el 85% de precisión (mAP50).")
    else:
        print("⚠️ El modelo no alcanzó el 85%. Considera aumentar las épocas o revisar la calidad del dataset.")


### Visualización de Curvas de Aprendizaje
Podemos ver cómo evolucionó la precisión y la pérdida (loss) durante el entrenamiento.


In [ ]:
# Mostrar la matriz de confusión
path_confusion = 'runs/detect/ppe_training_v1/confusion_matrix.png'
if os.path.exists(path_confusion):
    display(Image(filename=path_confusion, width=800))
else:
    print("La matriz de confusión no está disponible.")

# Mostrar gráficas de resultados (F1, P, R, etc.)
path_results = 'runs/detect/ppe_training_v1/results.png'
if os.path.exists(path_results):
    display(Image(filename=path_results, width=1000))


---

## 7. 🔮 Inferencia (Pruebas con datos nuevos)
Probemos el modelo con una imagen del conjunto de `test` que nunca ha visto.


In [ ]:
# Cargar el mejor modelo guardado
best_model_path = 'runs/detect/ppe_training_v1/weights/best.pt'
if os.path.exists(best_model_path):
    best_model = YOLO(best_model_path)
    
    # Intentar predecir en una imagen de la carpeta test
    test_img_dir = os.path.join(folder_dataset, 'test/images')
    if os.path.exists(test_img_dir):
        test_images = [f for f in os.listdir(test_img_dir) if f.endswith(('.jpg', '.png'))]
        if test_images:
            sample = os.path.join(test_img_dir, test_images[0])
            res = best_model.predict(sample, save=True, conf=0.5)
            print(f"Predicción realizada sobre: {sample}")
            # El resultado se guarda en runs/detect/predictX/
else:
    print("⚠️ No se encontró el archivo 'best.pt'.")


---

## 8. 📦 Exportación Final para Streamlit
Finalmente, movemos el archivo de pesos entrenados al directorio principal para que nuestra aplicación `main.py` pueda cargarlo.


In [ ]:
if os.path.exists(best_model_path):
    shutil.copy(best_model_path, 'modelo_ppe_yolov8.pt')
    print("✅ ¡ÉXITO! Modelo exportado como 'modelo_ppe_yolov8.pt'")
    
    # Respaldo en Google Drive si está disponible
    if USAR_DRIVE:
        try:
            drive_final = os.path.join(DRIVE_PROJECT_PATH, 'modelo_ppe_yolov8.pt')
            shutil.copy('modelo_ppe_yolov8.pt', drive_final)
            print(f"☁️ Modelo guardado permanentemente en Drive: {drive_final}")
        except Exception as e:
            print(f"⚠️ No se pudo copiar a Drive: {e}")

    # Si estás en Google Colab, esta línea intentará descargar el archivo automáticamente a tu PC
    try:
        from google.colab import files
        print("📥 Iniciando descarga del modelo a tu disco local...")
        files.download('modelo_ppe_yolov8.pt')
    except ImportError:
        print("ℹ️ No estás en Colab. El modelo se ha guardado en tu carpeta de proyecto local.")
        print(f"Ruta absoluta: {os.path.abspath('modelo_ppe_yolov8.pt')}")
    
    print("\nYa puedes usarlo en tu aplicación de Streamlit.")
else:
    print("❌ Error al exportar. No se encontraron los pesos del entrenamiento.")
